In [ ]:
!pip uninstall timm -y
!pip -q install --pre timm
!pip install fastai -qq

In [ ]:
import timm as timm
import os
import pandas as pd
from tqdm import tqdm
import cv2
import numpy as np
import matplotlib.pyplot as plt
from fastai.vision.all import *
from fastai.vision.data import ImageDataLoaders
from fastai.data.block import DataBlock, CategoryBlock
from fastai.data.transforms import RandomSplitter
from fastai.vision.data import ImageBlock
from fastai.vision.all import *
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

IMG_ROWS = 224
IMG_COLS = 224
NUM_CLASSES = 10
TEST_SIZE = 0.2
RANDOM_STATE = 99
NO_EPOCHS = 30
BATCH_SIZE = 128
CLASS_COUNT = 68
img_size = 224

path = os.getcwd()
train_dir = '/kaggle/input/competitions/image-processing-house-recognition/train/train'
test_dir = '/kaggle/input/competitions/image-processing-house-recognition/test/test'
df = pd.read_csv("/kaggle/input/competitions/image-processing-house-recognition/train.csv")

In [ ]:
new_train_dir = os.listdir(train_dir)
new_train_dir.sort()
new_train_dir[1:10]

In [ ]:
df = df.sort_values(by="image_name")
df

In [ ]:
df["class"].value_counts()

In [ ]:
# # Assuming train_dir, img_size, and imagenet_stats are defined
datablock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    splitter=RandomSplitter(seed=42),
    get_x = lambda row: os.path.join(train_dir, row['image_name']),
    get_y = lambda row: row['class'],
    # Apply the custom filter before converting to tensor and resizing.
    item_tfms=[ToTensor(), Resize(img_size)],
    batch_tfms=Normalize.from_stats(*imagenet_stats)
)
dls1 = datablock.dataloaders(df,shuffle_train = True, seed=123,bs=128)
dls1.train.show_batch(max_n=8)

In [ ]:
dls1.vocab

In [ ]:
avail_pretrained_models = timm.list_models('*eva*',pretrained=True)
len(avail_pretrained_models), avail_pretrained_models[:]

In [ ]:
# model_name = 'eva_giant_patch14_224.clip_ft_in1k'
model_name = 'eva02_base_patch14_224.mim_in22k'

In [ ]:
save_cb = SaveModelCallback(monitor='valid_loss',at_end=True,every_epoch=False)
early_stop_cb = EarlyStoppingCallback(monitor='valid_loss',min_delta=0.001 ,patience=10)
callbacks = [save_cb,early_stop_cb]

In [ ]:
learn = vision_learner(
    dls1, model_name,pretrained=True,
    path='/kaggle/working/',
    cbs=[ShowGraphCallback()] ,
    metrics=[accuracy]).to_fp16()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learn.model.to(device)  # Move model to one device first

In [ ]:
learn.model = torch.nn.DataParallel(learn.model)

In [ ]:
learn.fine_tune(10,freeze_epochs=3,cbs=callbacks)

In [ ]:
learn.validate()

In [ ]:
learn.show_results()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()
interp.print_classification_report()

In [ ]:
interp.plot_top_losses(k=10)

In [ ]:
learn.save('firstmodel')

In [ ]:
submission = pd.read_csv('/kaggle/input/image-processing-house-recognition/sample_submission.csv')

In [ ]:
submission.head()

In [ ]:
df_test=submission.copy()
df_test['id']=test_dir+'/'+df_test['id']+".jpg"
df_test

In [ ]:
learn.predict(df_test['id'][2])

In [ ]:
test_dl = learn.dls.test_dl(df_test['id'])

In [ ]:
preds, decoder = learn.get_preds(dl = test_dl)

In [ ]:
preds

In [ ]:
labels = np.argmax(preds, 1).tolist()

In [ ]:
# uncertainty = np.abs(preds[:, 0] - preds[:, 1])

# # Step 2: Sort by uncertainty and get the top 50 most uncertain indices
# top_50_uncertain_indices = np.argsort(uncertainty)[-1:]

# # Step 3: Create the labels where top 50 uncertain predictions are replaced with opposite argmax
# labels = np.argmax(preds, axis=1).tolist()  # Default labels using argmax

# for idx in top_50_uncertain_indices:
#     # Get the opposite of argmax (index of the smallest value)
#     opposite_idx = np.argmin(preds[idx])
#     labels[idx] = opposite_idx  # Replace with opposite class index

In [ ]:
len(labels)

In [ ]:
submission['answer'] = labels
submission['answer'] = [dls2.vocab[i] for i in submission['answer'] ]
submission

In [ ]:
submission.to_csv('submission_giant.csv',index=False)